<a href="https://colab.research.google.com/github/mbmanoj904-hue/Generative-ai/blob/main/CIT_Chat_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting data from multiple sources...")

# SOURCE A: Hardcoded Text / Paragraphs (The Detective Data)
raw_paragraphs = [
    "Witness A:Coorg Institute of Technology (CIT) is located in Ponnampet, Kodagu district, Karnataka. the principal of the institute is Dr. M  basavaraj",
    "Witness B:CIT offers undergraduate engineering programs, including Computer Science and Engineering (CSE).",
    "Witness The college conducts academic activities, technical events, workshops, seminars, and placement drives for its students."
]
# Crucial Step: Convert raw Python strings into LangChain Document objects
text_documents = [Document(page_content=text) for text in raw_paragraphs]

# SOURCE B: Web Scraping (The College Data)
urls = ["http://citcoorg.edu.in/admission/","https://www.citcoorg.edu.in/computer-science-engineering/","https://www.citcoorg.edu.in/"]
web_loader = WebBaseLoader(urls)
web_documents = web_loader.load()

# SOURCE C: PDF Documents
pdf_path = "CIT_BROCHURE.pdf" # Instruct students to upload a PDF with this name
try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception as e:
    print(f"Warning: PDF '{pdf_path}' not found in Colab. Skipping PDF ingestion.")
    pdf_documents = []

# MERGE ALL SOURCES
all_documents = text_documents + web_documents + pdf_documents
print(f"Total raw pages/documents loaded: {len(all_documents)}")

# Step 2: Data Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_documents)
print(f"Data chopped into {len(chunks)} searchable chunks.")

# Step 3: Embeddings & FAISS Vector Database
print("Building FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# Step 4: Initialize LLM
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("Groq")
)

print("\n🤖 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.")
print("-" * 60)

# Step 5: The Chat Loop
while True:
    user_query = input("\nYou: ")
    if user_query.lower() == 'exit':
        break

    # Semantic Search across ALL sources simultaneously
    relevant_chunks = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n\n".join([doc.page_content for doc in relevant_chunks])

    system_prompt = f"""
    You are an AI assistant powered by a multi-source knowledge base.
    Answer the user's question using ONLY the provided context, pdf, and the urls.

    Do NOT guess.

    CONTEXT:
    {retrieved_context}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )

    print(f"AI: {response.choices[0].message.content}")

Step 1: Ingesting data from multiple sources...
Total raw pages/documents loaded: 6
Data chopped into 115 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


🤖 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.
------------------------------------------------------------

You: what is the college name
AI: The college is named **CIT**.

You: where is college locate
AI: The college (CIT) is situated in the Kodagu district of Karnataka, India. Its campus spreads over about 31 acres and the institute is affiliated with Visvesvaraya Technological University, Belagavi.

You: what are the courses provide the college
AI: The college offers the following undergraduate engineering programmes:

- **Computer Science & Engineering (CSE)** – 120 seats  
- **Mechanical Engineering (ME)** – 30 seats  
- **Civil Engineering (CE)** – 30 seats  
- **Electronics & Communication Engineering (ECE)** – 60 seats  
- **Artificial Intelligence & Machine Learning (AI‑ML)** – 60 seats  
- **Artificial Intelligence & Data Science (AI‑DS)** – 60 seats  
- **Computer Science & Engineering – Cyber Security (CSE‑Cyber Security)** – 60 seats  

These